# Identifying Problem Stocks
In this notebook I will identify potential outlier stocks that show abnormal return 
trends with disproportional risk.

Black swan events such as the 2008 financial crisis, COVID-19, the dotcom bubble burst,
GameStop bull run may be legitimate reasons for outlier returns.

However, sometimes financial data may be corrupted for illegitimate reasons including,
double counting, acquisitions & mergers, bankruptcies etc.

It is important to identify potentially corrupted data so that it does not intefere with
algorithm testing and signal generation.

## Imports

In [ ]:
import pandas as pd
import numpy as np
import random
from matplotlib import pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from datetime import date
from dateutil.relativedelta import relativedelta

from trading_algos import optimization as tao
from trading_algos import datasets as tad
from trading_algos import plots as tap
from trading_algos import utils as tau
from trading_algos.utils import head_tail as ht

%load_ext autoreload
%autoreload 2

## Load Data

In [ ]:
# Load a complete collection of S&P500 stocks from repository
# Not interested in survivors here
df_stocks = tad.get_sp500(survivors=False).Close

In [ ]:
ht(df_stocks)

## Identifying Outliers

In [ ]:
def boxplot(title=''):

    fig, ax = plt.subplots(2,1, figsize=(12,6), tight_layout=True)
    plt.suptitle(title)

    ax[0].spines[['left','right']].set_visible(False)
    ax[0].tick_params(top=False,
                    bottom=False,
                    left=False,
                    right=False,
                    labelleft=False,
                    labelbottom=False)
    ax[0].set_ylabel('Risk', rotation=45, fontsize=12)
    sns.boxplot(df_agg[['Risk']], ax=ax[0], orient='h')

    ax[1].spines[['left','top','right']].set_visible(False)
    ax[1].tick_params(top=False,
                    bottom=False,
                    left=False,
                    right=False,
                    labelleft=False,
                    labelbottom=False)
    ax[1].set_ylabel('Return', rotation=45, fontsize=12)
    sns.boxplot(df_agg[['Return']], ax=ax[1], orient='h')

    plt.show();

In [ ]:
# Convert to log prices
log_prices = np.log(df_stocks)

# 3 Month Rolling Returns
log_returns = log_prices - log_prices.shift(63)

# Calculate Log Risk / Return
df_agg = log_returns.agg({'mean', 'std'}).rename({'mean': 'Return', 'std': 'Risk'}).T

# Annualize Log Risk / Return
df_agg = df_agg * (np.sqrt(4), 4)

# Get the upper and lower IQR bounds for outliers
outlier_bounds = df_agg.apply(tau.outlier_bounds, method='IQR')

# Identify stocks that sit beyond outlier thresholds for risk or return
df_outliers = df_agg[((df_agg > outlier_bounds.iloc[1]) | (df_agg < outlier_bounds.iloc[0]))\
                     .any(axis=1)]\
                        .sort_values(['Risk', 'Return'], ascending=False)

# Save list of tickers of potential outliers
outliers = df_outliers.index.to_list()

boxplot('There Are a Number of Extreme Outliers in Both Risk and Return')

In [ ]:
def trends(tickers,
           title='Log Price Trend of Outlier Stocks',
           txt='',
           ncols=5):
    
    ncols = min(ncols, len(tickers))
    nrows = int(np.ceil(len(tickers)/ncols))
    figlength = 1.5*nrows
    titlepad = 1 if txt != '' else 1
    padtop = figlength / (figlength + titlepad)

    fig, axes = plt.subplots(nrows, ncols, figsize=(12, figlength + titlepad), tight_layout=True)
    fig.tight_layout(pad=2, rect=[0, 0, 1, padtop])
    fig.suptitle(title, fontsize=16, y=1)
    fig.text(y=(1-(1-padtop)/2),
             x=0.5,
             s=txt,
             horizontalalignment='center',
             verticalalignment='center')
    
    for i, ticker in enumerate(tickers):

        if nrows > 1:
            ax_p = axes[int(i/ncols), i%ncols]
        else:
            ax_p = axes[i%ncols]

        ax_p.set_title(f"{ticker}")
        ax_p.set_yticklabels([])
        ax_p.set_yticks([])
        ax_p.set_ylabel(' ')
        ax_p.set_xticklabels([])
        ax_p.set_xticks([])
        ax_p.set_xlabel(' ')
        ax_p.spines[['top','bottom','left','right']].set_visible(False)

        ax_p.plot((log_prices[ticker]), linewidth=0.5)

    # Determine how many empty plots there are on the last row of the figure
    num_empty_plots = nrows * ncols - len(tickers)
    if num_empty_plots > 0:
        for i in range(1, num_empty_plots + 1):
            # Hide empty plots
            if nrows > 1:
                axes[nrows - 1, ncols - i].set_axis_off()
            else:
                axes[ncols-i].set_axis_off()

In [ ]:
txt = 'The majority of outlier stocks show smooth tends, indicating legitimate price '\
      'movement.\n Stocks such as CBE, TIE, CFC, MEE are showing a number sharp spikey '\
      'price movements indicating bad data, these stocks need to be isolated and removed.'

trends(tickers=outliers,
       txt=txt)

## Isolating The Bad Data

In [ ]:
# Daily Log Returns
log_returns = (log_prices - log_prices.shift(1))

#### Identifying Isolated Spikes

In [ ]:
# Abnormaly large price changes directly both preceded and followed by
# comparatively small price movements.
isolated_spikes = (
    # Abnormaly large daily change (> 170%)
    abs(log_returns > 1).mask(log_returns.isna(), np.nan) &
    # Followed by a comparatively smally daily change (< 10.5%)
    (abs(log_returns.shift(1) < 0.1).mask(log_returns.isna(), np.nan) &
    # Preceded by a comparitively small daily change (< 10.5%)
     abs(log_returns.shift(-1) < 0.1).mask(log_returns.isna(), np.nan))
    ).sum()

#### Identifying Spikes with Immediate Reversal

In [ ]:
# Abnormaly large price changes followed directly by an abnormaly large
# reversal.
reversal_spikes = (
    (((log_returns > 1) & (log_returns.shift(1) < -1)) | 
     ((log_returns < 1) & (log_returns.shift(1) > 1)))
).sum()

#### Identifying Large Residuals

In [ ]:
# Identifying all abnormal price movements that deviate enourmously
# from the monthly median price
rolling_med = log_returns.rolling(21).median()
residual_spikes = (abs(log_returns - rolling_med) > 2).sum()

#### Combining

In [ ]:
problem_counts = pd.concat(
    [isolated_spikes[isolated_spikes!=0],
     reversal_spikes[reversal_spikes!=0],
     residual_spikes[residual_spikes!=0]], 
    axis=1)\
        .fillna(0)\
            .astype(int)
problem_counts.columns = ['Isolated Spikes', 'Spike Reversals', 'Extreme Price Movements']
problem_tickers = problem_counts.index.tolist()
problem_counts

In [ ]:
txt = 'Stocks with extreme, questionable price movements have been isolated.\n' \
      'These stocks will be dropped from the dataset.   '

trends(tickers=problem_tickers,
       txt=txt)

## Removing Problem Stocks

In [ ]:
df_stocks = df_stocks.drop(problem_tickers, axis=1)

In [ ]:
# Convert to log prices
log_prices = np.log(df_stocks)

# 3 Month Rolling Returns
log_returns = log_prices - log_prices.shift(63)

# Calculate Log Risk / Return
df_agg = log_returns.agg({'mean', 'std'}).rename({'mean': 'Return', 'std': 'Risk'}).T

# Annualize Log Risk / Return
df_agg = df_agg * (np.sqrt(4), 4)

# Get the upper and lower IQR bounds for outliers
outlier_bounds = df_agg.apply(tau.outlier_bounds, method='IQR')

# Identify stocks that sit beyond outlier thresholds for risk or return
df_outliers = df_agg[((df_agg > outlier_bounds.iloc[1]) | (df_agg < outlier_bounds.iloc[0]))\
                     .any(axis=1)]\
                        .sort_values(['Risk', 'Return'], ascending=False)

# Save list of tickers of potential outliers
outliers = df_outliers.index.to_list()

boxplot('Extreme Outliers Have Been Removed. Remaining Outliers Are Legitimate.')

## Identifying Tickers with Intermittent Data

In [ ]:
def nulls_summary(data = pd.DataFrame):

    def get_nulls(s = pd.Series):
        
        start = s.first_valid_index()
        end = s.last_valid_index()
        
        if (start == None) | (end == None):
            return np.nan

        s_valid = s.loc[start: end].copy()
        
        nulls = s_valid.isna() * 1

        if nulls.sum() == 0:
            return np.nan
        
        nulls_periods =(
            ((nulls.diff().fillna(nulls.iloc[0]) == 1) * 1)\
                .mask(nulls==False,np.nan)\
                    .cumsum())
        
        stats = nulls_periods.value_counts()\
                    .agg(['count', 'mean', 'max', 'sum']).astype(int)
        
        stats['pct'] = nulls.mean() * 100

        stats.index=['Null Events', 'Avg Duration', 'Max Duration', 'Tot Missing', 'Pct Null']

        return stats.round(2)

    stats = data.apply(get_nulls)

    stats = stats[stats.notna()]

    df_null = pd.DataFrame(stats.tolist(), index=stats.index)

    df_null[df_null.columns[:4]] = df_null[df_null.columns[:4]].astype(int)

    return df_null

In [ ]:
df_nulls = nulls_summary(log_prices)
df_nulls

In [ ]:
# Dropping any tickers that have more than 5% of data missing
# or have atleast one period of 5 or more consecutive missing data points
problem_tickers = df_nulls[(df_nulls['Max Duration'] > 4) | (df_nulls['Pct Null'] >= 5)].index.tolist()

# Remaining tickers with missing data points will be smoothed out with
# the rolling weekly median
smooth_tickers = df_nulls.drop(problem_tickers).index.tolist()

In [ ]:
title = f'Log Price Trend of Outlier Stocks'
txt = 'The following tickers have extended periods of missing data and will be dropped.'

trends(tickers=problem_tickers,
        title=title,
        txt=txt,
        ncols=3)

In [ ]:
title = f'Log Price Trend of Outlier Stocks'
txt = 'The following tickers have short windows of missing data that may be smoothed.'

trends(tickers=smooth_tickers,
        title=title,
        txt=txt,
        ncols=5)

In [ ]:
plot_years = ['2021', '2022', '2023']

ncols = 2
nrows = 3

fig, ax = plt.subplots(nrows, ncols, figsize=(12, 6), tight_layout=True)
fig.tight_layout(pad=2, rect=[0, 0, 1, 0.90])
fig.suptitle('Smoothing Data Gaps With Weekly Rolling Median', fontsize=16, y=1)

for i, year in enumerate(plot_years):

    df_raw = df_stocks.loc[year, 'MHS'].copy()
    df_smoothed = df_raw.fillna(df_raw.rolling(5, min_periods=1).median())

    ax[i,0].set_yticklabels([])
    ax[i,0].set_yticks([])
    ax[i,0].set_ylabel(year, rotation=0, fontsize=10)
    ax[i,0].set_xticklabels([])
    ax[i,0].set_xticks([])
    ax[i,0].set_xlabel(' ')
    ax[i,0].spines[['top','bottom','left','right']].set_visible(False)
    ax[i,0].plot(df_raw)

    # Highlighing Periods of Missing Data
    ax[i,0].fill_between(df_raw.index, 
                         df_raw.min(), 
                         df_raw.max(), 
                         where=df_raw, 
                         facecolor="lightblue", 
                         alpha=0.3, 
                         label='Missing')
    
    ax[i,0].fill_between(df_raw.index, 
                         df_raw.min(), 
                         df_raw.max(), 
                         where=np.isfinite(df_raw), 
                         facecolor="white", alpha=1)


    ax[i,1].set_yticklabels([])
    ax[i,1].set_yticks([])
    ax[i,1].set_xticklabels([])
    ax[i,1].set_xticks([])
    ax[i,1].set_xlabel(' ')
    ax[i,1].spines[['top','bottom','left','right']].set_visible(False)
    ax[i,1].plot(df_smoothed)

    # Highlighing Periods of Missing Data
    ax[i,1].fill_between(df_smoothed.index, 
                         df_smoothed.min(), 
                         df_smoothed.max(), 
                         where=df_smoothed, 
                         facecolor="lightblue", 
                         alpha=0.3, 
                         label='Missing')
    
    ax[i,1].fill_between(df_smoothed.index, 
                         df_smoothed.min(), 
                         df_smoothed.max(), 
                         where=np.isfinite(df_smoothed), 
                         facecolor="white", alpha=1)
    
ax[0,0].set_title('Raw', fontsize=12)
ax[0,1].set_title('Smoothed', fontsize=12)
ax[0,0].legend(loc='upper left')
plt.show()

In [ ]:
# Smooth remaining stocks with the rolling weekly median
df_stocks = df_stocks.fillna(df_stocks.rolling(window=5, min_periods=1).median())